# PillSeek — in-house imprint reader (TrOCR fine-tune)
Runtime → Change runtime type → **A100 GPU** (Colab Pro). Run cells top to bottom. ~2–3 h.
Upload `manifest.json` in cell 2. Phone test photos are optional in cell 6.

In [ ]:
# 1) Install
!pip -q install transformers accelerate jiwer
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
# 2) Upload manifest.json
from google.colab import files
up = files.upload()
assert 'manifest.json' in up

In [ ]:
# 3) Download images (20-40 min)
import json, os, re, requests, hashlib
from concurrent.futures import ThreadPoolExecutor
rows = json.load(open('manifest.json', encoding='utf-8-sig'))
os.makedirs('imgs', exist_ok=True)
def norm_imprint(s):
    toks = [t for t in re.split(r'[;,\s]+', s.upper()) if t]
    return ' '.join(toks)
for r in rows:
    r['path'] = os.path.join('imgs', hashlib.md5(r['url'].encode()).hexdigest() + '.jpg')
    r['text'] = norm_imprint(r['imprint'])
rows = [r for r in rows if 1 <= len(r['text']) <= 40]
print(f'{len(rows)} labeled images')
def fetch(r):
    if os.path.exists(r['path']) and os.path.getsize(r['path']) > 0: return 0
    try:
        open(r['path'], 'wb').write(requests.get(r['url'], timeout=30).content); return 1
    except Exception: return -1
with ThreadPoolExecutor(16) as ex: res = list(ex.map(fetch, rows))
rows = [r for r in rows if os.path.exists(r['path']) and os.path.getsize(r['path']) > 1000]
print(f'downloaded {res.count(1)}, cached {res.count(0)}, failed {res.count(-1)}; usable {len(rows)}')

In [ ]:
# 4) Fine-tune TrOCR-large on pill imprints (60-120 min on A100)
import random, io
from PIL import Image, ImageFilter, ImageOps
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import Dataset, DataLoader

MODEL = 'microsoft/trocr-large-printed'
processor = TrOCRProcessor.from_pretrained(MODEL)
model = VisionEncoderDecoderModel.from_pretrained(MODEL).cuda()
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.generation_config.max_length = 24
model.generation_config.num_beams = 3

random.seed(0); random.shuffle(rows)
# Hold out whole pills (by slug) so eval measures reading, not memorizing.
slugs = sorted({r['slug'] for r in rows}); random.shuffle(slugs)
held = set(slugs[: max(200, len(slugs)//20)])
train_rows = [r for r in rows if r['slug'] not in held]
val_rows = [r for r in rows if r['slug'] in held]
print(f'train {len(train_rows)}  val {len(val_rows)} (held-out pills: {len(held)})')

def phone_aug(img):
    if random.random() < 0.5:
        w = random.randint(320, 700); img = img.resize((w, int(w*img.height/img.width)), Image.BILINEAR)
    if random.random() < 0.4: img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 1.5)))
    if random.random() < 0.5:
        buf = io.BytesIO(); img.save(buf, 'JPEG', quality=random.randint(35, 85)); img = Image.open(io.BytesIO(buf.getvalue())).convert('RGB')
    if random.random() < 0.5: img = img.rotate(random.uniform(-25, 25), fillcolor=(128,128,128), expand=False)
    if random.random() < 0.5: img = ImageOps.autocontrast(img, cutoff=random.randint(0, 5))
    if random.random() < 0.3:
        from PIL import ImageEnhance
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.6, 1.4))
    return img

class DS(Dataset):
    def __init__(self, rs, train): self.rs, self.train = rs, train
    def __len__(self): return len(self.rs)
    def __getitem__(self, i):
        r = self.rs[i]
        img = Image.open(r['path']).convert('RGB')
        if self.train: img = phone_aug(img)
        pv = processor(images=img, return_tensors='pt').pixel_values[0]
        labels = processor.tokenizer(r['text'], padding='max_length', max_length=24, truncation=True).input_ids
        labels = [l if l != processor.tokenizer.pad_token_id else -100 for l in labels]
        return pv, torch.tensor(labels)

dl = DataLoader(DS(train_rows, True), batch_size=24, shuffle=True, num_workers=4, drop_last=True)
opt = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
EPOCHS = 4
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-5, total_steps=EPOCHS*len(dl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda')
model.train()
for ep in range(EPOCHS):
    tot = 0.0
    for step, (pv, labels) in enumerate(dl):
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            loss = model(pixel_values=pv.cuda(), labels=labels.cuda()).loss
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        tot += loss.item()
        if (step+1) % 100 == 0: print(f'epoch {ep+1} step {step+1}/{len(dl)} loss {tot/(step+1):.3f}')
    print(f'=== epoch {ep+1} avg loss {tot/len(dl):.3f} ===')
model.save_pretrained('pill_trocr'); processor.save_pretrained('pill_trocr')
print('saved pill_trocr/')

In [ ]:
# 5) Accuracy on held-out pills (never seen in training)
import jiwer
model.eval()
def read(img):
    pv = processor(images=img, return_tensors='pt').pixel_values.cuda()
    with torch.no_grad(): ids = model.generate(pv)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip().upper()
exact = tok_hits = tok_total = 0
sample = val_rows[:600]
for r in sample:
    pred = read(Image.open(r['path']).convert('RGB'))
    if pred == r['text']: exact += 1
    gt = set(r['text'].split()); pr = set(pred.split())
    tok_hits += len(gt & pr); tok_total += len(gt)
print(f'Held-out catalog images: exact imprint {100*exact/len(sample):.1f}%   token recall {100*tok_hits/tok_total:.1f}%   (n={len(sample)})')
for r in sample[:8]: print(f"  truth: {r['text']:<20} read: {read(Image.open(r['path']).convert('RGB'))}")

In [ ]:
# 6) Optional: read your own phone photos (upload any number, e.g. Augmentin / Jardiance shots)
from google.colab import files
ups = files.upload()
for name in ups:
    img = Image.open(name).convert('RGB')
    w, h = img.size; s = int(min(w, h) * 0.6)  # center crop, like a 'fit the pill' guide
    crop = img.crop(((w-s)//2, (h-s)//2, (w+s)//2, (h+s)//2))
    print(f'{name}: full-frame -> {read(img)!r}   center-crop -> {read(crop)!r}')

In [ ]:
# 7) Package + download (model ~1.3GB). Use the Files sidebar if the download button is blocked.
!zip -qr pill_trocr.zip pill_trocr
from google.colab import files
files.download('pill_trocr.zip')